# LIFE on Google Colab — PolitiFact++ (binary MF-vs-MR, LLaMA2-7B)

Faithful reproduction of the paper's setup: binary fake/real over the **LLM pair** (MF=fake, MR=real), with **LLaMA2-7B** as the reconstruction model. End-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. Step 3 uses the **ungated** `NousResearch/Llama-2-7b-hf` mirror by default — no HF token needed. (The official `meta-llama/Llama-2-7b-hf` is gated and requires an approved access request + token.)

Scope: **PolitiFact++** only (~229 LLM-pair articles: 97 fake + 132 real). VLPFN is excluded (its text has no punctuation, so sentence splitting cannot work). GossipCop++ is far heavier; try it only after this works.

In [3]:
# Confirm a GPU is attached
!nvidia-smi

Thu Jun 18 00:43:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/PolitiFact++'

# Paper's binary task: LLM pair only (MF=fake, MR=real), reconstructed with LLaMA2-7B.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/output_bin'        # MF_fake.jsonl + MR_true.jsonl
KEY_SENT       = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top10.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/bert_bin.pt'       # fresh extractor for MF-vs-MR
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/features_llama'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/train_bin.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/test_bin.jsonl'

print('cwd:', os.getcwd())
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))

cwd: /content/drive/MyDrive/LIFE
PolitiFact++ found: True


In [6]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.7/644.7 kB 36.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.42.0 requires rich<14,>=12.4.4, but you have rich 11.2.0 which is incompatible.
pymc 5.28.5 requires rich>=13.7.1, but you have rich 11.2.0 which is incompatible.


In [7]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## HuggingFace login (optional)
Step 3 defaults to the **ungated** `NousResearch/Llama-2-7b-hf` mirror, so **no token is needed — you can skip this cell**. Only run it if you switch Step 3 to the official gated `meta-llama/Llama-2-7b-hf` (which also requires an approved access request).

In [8]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
#from huggingface_hub import login
#login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Step 0 — Convert PolitiFact++ to the binary LLM-pair JSONL
`--subset llm` emits only `MF_fake.jsonl` (97, fake) and `MR_true.jsonl` (132, real) — the paper's binary task. HF/HR (human-written) are not used.

In [9]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_BIN}" --subset llm

MF.json -> MF_fake.jsonl: 97 records (label=gpt3.5_fake)
MR.json -> MR_true.jsonl: 132 records (label=gpt3.5_true)


## Step 1 — Key-sentence extraction (top-10)
Trains a **fresh** BERT fake/real classifier on MF-vs-MR (saved to `BERT_CKPT`), then keeps the **top-10** most impactful sentences per article (paper's k for PolitiFact++). This is the slowest step (a forward pass per sentence per article).

In [10]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 221kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 757kB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 1.37MB/s]
config.json: 100% 570/570 [00:00<00:00, 2.21MB/s]
model.safetensors: 100% 440M/440M [00:03<00:00, 142MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 6204.87it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 


## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [11]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

所有 .jsonl 文件已成功更新。


## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`.

In [12]:
# meta-llama/Llama-2-7b-hf is gated (needs Meta approval). NousResearch/Llama-2-7b-hf is an
# ungated mirror of the SAME weights/tokenizer — no token needed. Swap back to the official
# repo if/when your access request is approved.
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

device: cuda | model: NousResearch/Llama-2-7b-hf | dtype: bfloat16 | scorer: llama
config.json: 100% 583/583 [00:00<00:00, 3.10MB/s]
tokenizer_config.json: 100% 746/746 [00:00<00:00, 4.22MB/s]
tokenizer.model: 100% 500k/500k [00:01<00:00, 467kB/s]
tokenizer.json: 100% 1.84M/1.84M [00:00<00:00, 144MB/s]
special_tokens_map.json: 100% 435/435 [00:00<00:00, 3.39MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 26.8k/26.8k [00:00<00:00, 80.3MB/s]
Fetching 2 files: 100% 2/2 [00:38<00:00, 19.08s/it]
Download complete: 100% 13.5G/13.5G [00:38<00:00, 353MB/s]
Loading weights: 100% 291/291 [00:02<00:00, 108.35it/s]
generation_config.json: 100% 200/200 [00:00<00:00, 1.37MB/s]
input file:/content/drive/MyDrive/LIFE/dataset/output_bin/MF_fake.jsonl, length:97
  0% 0/97 [00:00<?, ?it/s]0 57
58 122
124 202
206 280
282 306
310 350
352 436
438 484
486 538
542 578
580 648
  1% 1/97 [00:00<01:03,  1.51it/s]0 57
58 122
124 158
161 197
199 247
250 294

## Step 4A — train the classifier (released head: BMES tags + CRF + majority vote)
Splits `FEATURES_LLAMA` into train/test (seed-0, deterministic) and trains the released Transformer classifier for **50 epochs** on the binary MF-vs-MR task. This head diverges from the paper — it tags tokens with B/M/E/S labels, CRF-decodes them, and recovers the article label by majority vote — so it is the **A-side** of the head A/B. Step 4B below is the paper-faithful head. Paper target for PolitiFact++: **Acc 0.900 / F1 0.882**.

In [13]:
#!python LIFE_train/train.py \
 # --split_dataset \
  #--data_path "{FEATURES_LLAMA}" \
  #--train_path "{TRAIN_PATH}" \
  #--test_path "{TEST_PATH}" \
  #--model Transformer \
  #--num_train_epochs 50

Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1128, in get_code
  File "<frozen importlib._bootstrap_external>", line 757, in _compile_bytecode
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/MyDrive/LIFE/LIFE_train/train.py", line 14, in <module>
    from transformers.optimization import get_linear_schedule_with_warmup
  File "/usr/local/lib/python3.12/dist-packages/transformers/optimization.py", line 28, in <module>
    from .trainer_utils import SchedulerType
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer_utils.py", line 66, in <module>
    from peft import PeftMixedModel, PeftModel
  File "/usr/local/lib/p

## Step 4B — paper head (sigmoid + BCE, Eq 11–12)
Same CNN→Transformer trunk, but the head matches the paper: masked mean-pool → one sigmoid probability per article, trained with **binary cross-entropy** (fake=1, real=0) and evaluated directly at article level — no BMES tags, no CRF, no majority vote. Files: `LIFE_train/model_bce.py` + `LIFE_train/train_bce.py` (the originals are untouched and remain the A-side).

**The A/B is fair:** 4A and 4B consume the *same* `FEATURES_LLAMA` and the *same* seed-0 train/test split (`--split_dataset` here regenerates the identical split, so running 4A first is not required). The test set is only ~46 articles (≈2 accuracy points per article), so **re-run this cell with `--seed 1`, `2`, `3`, `4` and report mean ± std**. A-side references: 85.5/80.8 and 83.9/78.2; paper target 90.0/88.2. Checkpoint: `bce_en.pt`.

In [36]:
!python LIFE_train/train_bce.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --num_train_epochs 50 \
  --seed 20

Log INFO: split dataset...
********************************
The overall data sources:
['MF_fake.jsonl', 'MR_true.jsonl']
100% 183/183 [00:00<00:00, 3735.71it/s]
100% 46/46 [00:00<00:00, 3438.26it/s]

The number of train dataset: 183
The number of test  dataset: 46
********************************
100% 183/183 [00:00<00:00, 7130.78it/s]
100% 46/46 [00:00<00:00, 8430.03it/s]
seed: 20
--------------------------------BCE head (paper Eq 11-12)--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/6 [00:00<?, ?it/s]
Iteration:  17% 1/6 [00:00<00:03,  1.66it/s]
Iteration:  33% 2/6 [00:00<00:01,  3.16it/s]
Iteration:  50% 3/6 [00:00<00:00,  4.48it/s]
Iteration:  67% 4/6 [00:00<00:00,  5.72it/s]
Iteration: 100% 6/6 [00:01<00:00,  5.37it/s]
epoch 1: train_loss 0.7059244910875956

Iteration:   0% 0/2 [00:00<?, ?it/s]
Iteration: 100% 2/2 [00:00<00:00, 13.33it/s]
******** Evalation ********
Accuracy: 63.0
Macro F1 Score: 38.7
Precision/Recall per 

## Notes / troubleshooting
- **HF gating**: Step 3 defaults to the ungated `NousResearch/Llama-2-7b-hf` mirror (no token). If you switch to the official `meta-llama` repo and hit a 403 "gated repo", your access request hasn't been approved yet.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.